## Gabriel Scaraficci de Lima - rm563739

## Karen Akemi de Oliveira - rm562733

#Primeiros passos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from google.colab import data_table
data_table.enable_dataframe_formatter()

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/diogenesjusto/FIAP/master/Gradua%C3%A7%C3%A3o/dados/Fraud_Study_Enrich.csv")
df.head(3)

,Unnamed: 0,X,ID_CLIENTE,COD_PARCEIRO_NEGOCIO,COD_NOTA_SERVICO,COD_EMPRESA,DT_MES_EXECUCAO,DESC_MUNICIPIO,DESC_BAIRRO,DESC_CLASSE_CALCULO,...,RESULTADO,SEXO,FaixaIDade,OBITO,ESCOLARIDADE,RENDAESTIMADA,FAIXARENDA,SCORE1,SCORE2,RESULTADO_N
0,1,1,73fe6addf5437873212b29a12c981c15b0db8c06574e61...,700614111,716573134,D002,201412,VOTORANTIM,JD NOVO MUNDO,Residencial,...,IRREGULAR,F,I - Acima de 75 anos,NAO,NaN,380.0,E - DE 0 A 2 SM,10.0,NaN,1
1,2,2,d73cf2fb49fb6538d5c3ade7998b4bacc838b8160e0924...,710914826,715897522,D001,201410,BIRIGUI,CJH JOAO CREVELARO,Residencial,...,REGULAR,F,I - Acima de 75 anos,NAO,NaN,545.0,E - DE 0 A 2 SM,9.0,10.0,0
2,3,3,72afa50efa52af6587f7bbdb486e52b8ecbb91c51c88c9...,701963566,716088410,D001,201411,CAMPINAS,VL PROOST DE SOUZA,Residencial,...,REGULAR,M,E - De 36 a 45 anos,NAO,"COLEGIAL COMPLETO, OU MEDIO COMPLETO",1024.0,E - DE 0 A 2 SM,10.0,8.0,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20502 entries, 0 to 20501
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            20502 non-null  int64  
 1   X                     20502 non-null  int64  
 2   ID_CLIENTE            20502 non-null  object 
 3   COD_PARCEIRO_NEGOCIO  20502 non-null  int64  
 4   COD_NOTA_SERVICO      20502 non-null  int64  
 5   COD_EMPRESA           20502 non-null  object 
 6   DT_MES_EXECUCAO       20502 non-null  int64  
 7   DESC_MUNICIPIO        20502 non-null  object 
 8   DESC_BAIRRO           20502 non-null  object 
 9   DESC_CLASSE_CALCULO   20502 non-null  object 
 10  DT_REFERENCIA         20502 non-null  int64  
 11  NUM_DIAS              20502 non-null  int64  
 12  FATURADO              20502 non-null  int64  
 13  RESULTADO             20502 non-null  object 
 14  SEXO                  20502 non-null  object 
 15  FaixaIDade         

#Análise descritiva

## Fraudes

In [ ]:
#Quantidade de fraudes
df['RESULTADO_N'].value_counts()

,count
RESULTADO_N,
0,16666
1,3836


In [ ]:
#Percentual de fraudes
df['RESULTADO_N'].value_counts(normalize=True) * 100

,proportion
RESULTADO_N,
0,81.28963
1,18.71037


### Insight:
Observa-se que o dataset está desbalanceado, apresentando maior quantidade de casos não fraudulentos, porém uma quantidade considerável de fraudes

##Hipótese 1 - COD_EMPRESA


Algumas empresas podem apresentar maior concentração de fraudes

In [ ]:
# Quantidade de fraudes por empresa
empresa = pd.crosstab(df['COD_EMPRESA'], df['RESULTADO_N'])
empresa['TAXA_FRAUDE'] = (empresa[1] / empresa.sum(axis=1)) * 100
print(empresa.sort_values('TAXA_FRAUDE',ascending=False).round(2))

RESULTADO_N     0     1  TAXA_FRAUDE
COD_EMPRESA                         
D005           16    11        40.74
D003          100    28        21.88
D001         8683  2360        21.37
D002         7856  1436        15.45
D004            7     1        12.50
D006            2     0         0.00
D007            2     0         0.00


## Hipótese 2 - DESC_MUNICIPIO

Algumas cidades podem apresentar incidência maior de fraude

In [ ]:
# Quantidade de fraudes por município
mun = pd.crosstab(df['DESC_MUNICIPIO'], df['RESULTADO_N'])
mun['TAXA_FRAUDE'] = (mun[1]/mun.sum(axis=1)) * 100
print(mun.sort_values('TAXA_FRAUDE',ascending=False).round(2))

RESULTADO_N      0  1  TAXA_FRAUDE
DESC_MUNICIPIO                    
ALTINOPOLIS      0  4        100.0
ARANDU           0  2        100.0
BENTO DE ABREU   0  5        100.0
BOCAINA          0  1        100.0
BARRA DO JACARE  0  2        100.0
...             .. ..          ...
SOCORRO          2  0          0.0
TEJUPA           1  0          0.0
TABATINGA        1  0          0.0
UBARANA          6  0          0.0
VIRADOURO        1  0          0.0

[190 rows x 3 columns]


##Hipótese 3 - FAIXARENDA

Pessoas de baixa renda podem ter alguma influência em padrões de fraude

In [ ]:
# Quantidade de fraudes por renda
renda = pd.crosstab(df['FAIXARENDA'], df['RESULTADO_N'])
renda['TAXA_FRAUDE'] = (renda[1]/renda.sum(axis=1)) * 100
print(renda.sort_values('TAXA_FRAUDE', ascending=False).round(2))

RESULTADO_N             0     1  TAXA_FRAUDE
FAIXARENDA                                  
E - DE 0 A 2 SM     13445  3216        19.30
D - DE 2 A 4 SM      2205   431        16.35
C - DE 4 A 10 SM      831   159        16.06
B - DE 10 A 20 SM     139    23        14.20
A - ACIMA DE 20 SM     46     7        13.21


##Hipótese 4 - FaixaIDade

Pessoas mais velhas podem ser mais vulneráveis a fraudes

In [ ]:
# Quantidade de fraudes por idade
idade = pd.crosstab(df['FaixaIDade'], df['RESULTADO_N'])
idade['TAXA_FRAUDE'] = (idade[1]/idade.sum(axis=1)) * 100
print(idade.sort_values('TAXA_FRAUDE', ascending=False).round(2))

RESULTADO_N              0     1  TAXA_FRAUDE
FaixaIDade                                   
I - Acima de 75 anos  1707   556        24.57
E - De 36 a 45 anos   2868   681        19.19
J - Indefinido          90    21        18.92
H - De 66 a 75 anos   1449   327        18.41
F - De 46 a 55 anos   5620  1245        18.14
G - De 56 a 65 anos   3049   661        17.82
D - De 26 a 35 anos   1844   344        15.72
C - De 18 a 25 anos     39     1         2.50


##Hipótese 5 - OBITO

Podem ser usados dados de pessoas falecidas para fraudes

In [ ]:
# Quantidade de fraude por obito
obito = pd.crosstab(df['OBITO'], df['RESULTADO_N'])
obito['TAXA_FRAUDE'] = (obito[1]/obito.sum(axis=1)) * 100
print(obito.sort_values('TAXA_FRAUDE', ascending=False).round(2))

RESULTADO_N      0     1  TAXA_FRAUDE
OBITO                                
SIM            374   120        24.29
NAO          16292  3716        18.57


##Hipótese 6 - DT_MES_EXECUCAO

Fraudes podem apresentar sazonalidade

In [ ]:
# Qauntidade de fraude por mês
mes = pd.crosstab(df['DT_MES_EXECUCAO'], df['RESULTADO_N'])
mes['TAXA_FRAUDE'] = (mes[1]/mes.sum(axis=1)) * 100
print(mes.sort_values('TAXA_FRAUDE', ascending=False).round(2))

RESULTADO_N         0    1  TAXA_FRAUDE
DT_MES_EXECUCAO                        
201410           3170  816        20.47
201411           2846  712        20.01
201409           2665  628        19.07
201407           2156  487        18.43
201408           2209  477        17.76
201412           3620  716        16.51


# Modelo base

In [ ]:
#  Separação de bases de treino e teste
x_treino, x_teste, y_treino, y_teste = train_test_split(df[['COD_NOTA_SERVICO', 'DT_MES_EXECUCAO']],
                                                        df['RESULTADO_N'], test_size=0.32, random_state=25)

In [ ]:
# Modelo de classificação (neste caso, DecisionTree)
mod = tree.DecisionTreeClassifier()
mod = mod.fit(x_treino, y_treino)

In [ ]:
# Executando a previsão
y_prev = mod.predict(x_teste)

In [ ]:
# Matriz de confusão
pd.crosstab(y_prev, y_teste, margins=True)

RESULTADO_N,0,1,All
row_0,,,
0,4886,438,5324
1,432,805,1237
All,5318,1243,6561


In [ ]:
print(accuracy_score(y_teste, y_prev))
print(precision_score(y_teste, y_prev))
print(recall_score(y_teste, y_prev))

0.8678555098308185
0.6518578352180937
0.6492357200321802


In [ ]:
# Verificando a importância de cada variável no resultado
importancias = pd.DataFrame({
    'Variavel': x_treino.columns,
    'Importancia': mod.feature_importances_
})

print(importancias.sort_values(by='Importancia', ascending=False))

           Variavel  Importancia
0  COD_NOTA_SERVICO     0.960347
1   DT_MES_EXECUCAO     0.039653


**Não fraudes (0):**

Acertos: 91,77%

Erros: 8,23%


**Fraudes (1):**

Acertos: 65,08

Erros: 34,92%

# Modelo 1 - DecisionTree

In [ ]:
# Tranformando as string em 1 ou 0

encoder = LabelEncoder()
colunas = [
    'COD_NOTA_SERVICO',
    'DESC_MUNICIPIO',
    'FAIXARENDA'
]

for coluna in colunas:
  df[coluna] = encoder.fit_transform(df[coluna])

In [ ]:
# Definindo as variáveis
x = df[colunas]
y = df['RESULTADO_N']

In [ ]:
# Treino e teste
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.25, random_state=25)

In [ ]:
# Definindo modelo
modelo = DecisionTreeClassifier()
modelo.fit(x_train, y_train)

DecisionTreeClassifier()

In [ ]:
# Executano previsão
y_pred = modelo.predict(x_test)

In [ ]:
# Matriz de confusão
pd.crosstab(y_pred, y_test, margins=True)

RESULTADO_N,0,1,All
row_0,,,
0,3880,317,4197
1,275,654,929
All,4155,971,5126


In [ ]:
# Verificando resultados
print('accuracy: ', accuracy_score(y_test, y_pred))
print('precision: ', precision_score(y_test, y_pred))
print('recall: ', recall_score(y_test, y_pred))

accuracy:  0.8845103394459618
precision:  0.7039827771797632
recall:  0.6735324407826982


In [ ]:
# Verificando a importância de cada variável no resultado
importancias = pd.DataFrame({
    'Variavel': x_train.columns,
    'Importancia': modelo.feature_importances_
})

print(importancias.sort_values(by='Importancia', ascending=False))

           Variavel  Importancia
0  COD_NOTA_SERVICO     0.808359
1    DESC_MUNICIPIO     0.161565
2        FAIXARENDA     0.030076


**Não fraudes (0):**

Acertos: 92,51%

Erros: 7,49%


**Fraudes (1):**

Acertos: 70,42%

Erros: 29,58%

# Modelo 2 - DecisionTree

In [ ]:
# Tranformando as string em 1 ou 0 - encoding
encoder = LabelEncoder()
colunas = [
    'COD_NOTA_SERVICO',
    'DESC_MUNICIPIO',
    'FaixaIDade'
]

for coluna in colunas:
  df[coluna] = encoder.fit_transform(df[coluna])

In [ ]:
# Definindo as variáveis
x = df[colunas]
y = df['RESULTADO_N']

In [ ]:
# Treino e teste
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.20, random_state=25)

In [ ]:
# Definindo modelo
modelo = DecisionTreeClassifier()
modelo.fit(x_train, y_train)

DecisionTreeClassifier()

In [ ]:
# Executando previsão
y_pred = modelo.predict(x_test)

In [ ]:
# Matriz de confusão
pd.crosstab(y_pred, y_test, margins=True)

RESULTADO_N,0,1,All
row_0,,,
0,3092,246,3338
1,220,543,763
All,3312,789,4101


In [ ]:
# Verificando resultados
print('accuracy: ', accuracy_score(y_test, y_pred))
print('precision: ', precision_score(y_test, y_pred))
print('recall: ', recall_score(y_test, y_pred))

accuracy:  0.8863691782492075
precision:  0.7116644823066841
recall:  0.688212927756654


In [ ]:
# Verificando a importância de cada variável no resultado
importancias = pd.DataFrame({
    'Variavel': x_train.columns,
    'Importancia': modelo.feature_importances_
})

print(importancias.sort_values(by='Importancia', ascending=False))

           Variavel  Importancia
0  COD_NOTA_SERVICO     0.741040
1    DESC_MUNICIPIO     0.167647
2        FaixaIDade     0.091313


**Não fraudes (0):**

Acertos: 92,63%

Erros: 7,37%


**Fraudes (1):**

Acertos: 71,17%

Erros: 28,83%

# Modelo 3 - RandomForest

In [ ]:
# Tranformando as string em 1 ou 0

encoder = LabelEncoder()
colunas = [
    'COD_NOTA_SERVICO',
    'DESC_MUNICIPIO',
    'FaixaIDade',
    'FAIXARENDA'
]

for coluna in colunas:
  df[coluna] = encoder.fit_transform(df[coluna])

In [ ]:
# Definindo as variáveis
x = df[colunas]
y = df['RESULTADO_N']

In [ ]:
# Treino e teste
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.25, random_state=25)

In [ ]:
modelo_rf = RandomForestClassifier()

In [ ]:
modelo_rf.fit(x_train, y_train)

RandomForestClassifier()

In [ ]:
y_pred_rf = modelo_rf.predict(x_test)

In [ ]:
pd.crosstab(y_pred_rf, y_test, margins=True)

RESULTADO_N,0,1,All
row_0,,,
0,3940,358,4298
1,215,613,828
All,4155,971,5126


In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Precision:', precision_score(y_test, y_pred_rf))
print('Recall:', recall_score(y_test, y_pred_rf))

Accuracy: 0.888216933281311
Precision: 0.7403381642512077
Recall: 0.631307929969104


In [ ]:
# Verificando a importância de cada variável no resultado
importancias = pd.DataFrame({
    'Variavel': x_train.columns,
    'Importancia': modelo_rf.feature_importances_
})

print(importancias.sort_values(by='Importancia', ascending=False))

           Variavel  Importancia
0  COD_NOTA_SERVICO     0.721499
1    DESC_MUNICIPIO     0.186106
2        FaixaIDade     0.067485
3        FAIXARENDA     0.024910


**Não fraudes (0):**

Acertos: 91,67%

Erros: 8,33%


**Fraudes (1):**

Acertos: 74,03%

Erros: 25,97%

# Escolha de modelo

## Comparação dos modelos

$$
\begin{array}{|c|c|c|c|}
\hline
\text{} & \text{Modelo base} & \text{Modelo 1} & \text{Modelo 2} & \text{Modelo 3}\\
\hline
\text{Accuracy} & 0.87 & 0.88 & 0.89 & 0.89\\
\text{Precision} & 0.65 & 0.70 & 0.71 & 0.74 \\
\text{Recall} & 0.65 & 0.67 & 0.69 & 0.63 \\
\hline
\end{array}
$$

### Acertos e falsos positivos (FP)

$$
\begin{array}{|c|c|c|c|}
\hline
\text{} & \text{Modelo base} & \text{Modelo 1} & \text{Modelo 2} & \text{Modelo 3}\\
\hline
\text{Acertos} & 91.77\% & 92,51\% & 92.63\% & 91,67\%\\
\text{Erros} & 8.23\% & 7.49\% & 7.37\% & 8.33\% \\
\hline
\end{array}
$$

### Acertos e falsos negativos (FN)

$$
\begin{array}{|c|c|c|c|}
\hline
\text{} & \text{Modelo base} & \text{Modelo 1} & \text{Modelo 2} & \text{Modelo 3}\\
\hline
\text{Acertos} & 65.08\% & 70.42\% & 71.17\% & 74.03\% \\
\text{Erros} & 34.92\% & 29.58\% & 28.83\% & 25.97\% \\
\hline
\end{array}
$$

### Conclusão

O **modelo 3**, usando Random Forest, apresentou melhores resultado em relação aos outros modelos

Por mais que tenha tido o menor recall, obteve melhores resultados no accuracy e no precision e, além disso, o FP e FN apresentaram a melhor combinação de resultados